In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
CHEMIN = "data/instagram_users_lifestyleSafe.csv"

Matplotlib is building the font cache; this may take a moment.


In [3]:
df = pd.read_csv(CHEMIN)
print(df.shape)

(200000, 58)


## Choix des colonnes

**Filtre 1 – Le droit.** Nous ne gardons que les colonnes qu'Instagram connaît pour tous ses utilisateurs et qu'il peut légalement utiliser pour du ciblage. Sont exclues : les données de santé et de bien-être (article 9 du RGPD), les loisirs et la situation sociale (déclarés dans l'enquête du panel uniquement), et les identifiants (`user_id`, `app_name`). Le stress, le bonheur et le sommeil serviront uniquement à décrire les segments.

In [5]:
CANDIDATES = [
    "daily_active_minutes_instagram", "sessions_per_day", "average_session_length_minutes",
    "posts_created_per_week", "reels_watched_per_day", "stories_viewed_per_day",
    "likes_given_per_day", "comments_written_per_day", "dms_sent_per_week", "dms_received_per_week",
    "ads_viewed_per_day", "ads_clicked_per_day",
    "time_on_feed_per_day", "time_on_explore_per_day", "time_on_messages_per_day", "time_on_reels_per_day",
    "followers_count", "following_count", "notification_response_rate", "linked_accounts_count",
    "user_engagement_score", "account_creation_year",
]
print(len(CANDIDATES), "colonnes candidates après le filtre 1")

22 colonnes candidates après le filtre 1


In [6]:
pd.set_option("display.max_columns", None)

filtre2 = pd.DataFrame({
    "valeurs_differentes": df[CANDIDATES].nunique(),
    "corr_temps_passe":    df[CANDIDATES].corrwith(df["daily_active_minutes_instagram"]).round(2),
    "corr_stress":         df[CANDIDATES].corrwith(df["perceived_stress_score"]).round(2),
    "corr_bonheur":        df[CANDIDATES].corrwith(df["self_reported_happiness"]).round(2),
}).sort_values("corr_temps_passe", ascending=False)

filtre2

,valeurs_differentes,corr_temps_passe,corr_stress,corr_bonheur
daily_active_minutes_instagram,532,1.00,0.83,-0.37
likes_given_per_day,306,0.98,0.82,-0.37
time_on_feed_per_day,305,0.97,0.81,-0.36
stories_viewed_per_day,138,0.95,0.81,-0.35
comments_written_per_day,81,0.94,0.79,-0.35
time_on_reels_per_day,203,0.93,0.78,-0.35
dms_received_per_week,89,0.92,0.77,-0.35
dms_sent_per_week,81,0.90,0.75,-0.34
time_on_messages_per_day,124,0.90,0.75,-0.33
ads_viewed_per_day,51,0.89,0.75,-0.33


**Filtre 2 – L'utilité statistique.** Aucune colonne candidate n'est constante (toutes ont au moins 6 valeurs différentes). Le tableau révèle quatre familles :

- **Copies du temps passé (corrélation ≥ 0,87)** : likes, temps sur le fil, stories, commentaires, temps sur les Reels, messages reçus et envoyés, temps en messagerie, publicités vues, temps sur Explorer, Reels regardés. Ces colonnes répètent la même information que le temps passé. Nous gardons `daily_active_minutes_instagram` pour représenter la quantité d'usage. Les autres serviront à construire des ratios qui décrivent le style d'usage.
- **Colonnes en partie liées (0,50 à 0,75)** : sessions par jour, publicités cliquées, publications par semaine. Elles contiennent une part d'information nouvelle et sont conservées pour les ratios.
- **Colonnes indépendantes (corrélation < 0,20)** : durée moyenne d'une session, abonnés, abonnements, année de création du compte, taux de réponse aux notifications, comptes liés. Elles décrivent autre chose que la quantité d'usage, mais aucune n'est liée au stress (corrélation < 0,20).
- **Colonne suspecte** : `user_engagement_score` baisse quand le temps passé augmente (corrélation de −0,53). Ce comportement est contraire à ce qu'on attend d'un score d'engagement, et son mode de calcul n'est pas documenté. Elle est exclue.

In [8]:
temps_total = df[["time_on_feed_per_day", "time_on_explore_per_day",
                  "time_on_messages_per_day", "time_on_reels_per_day"]].sum(axis=1)

df["part_reels"]         = df["time_on_reels_per_day"] / temps_total
df["part_messages"]      = df["time_on_messages_per_day"] / temps_total
df["part_explore"]       = df["time_on_explore_per_day"] / temps_total
df["ratio_createur"]     = (df["posts_created_per_week"] / 7 + df["comments_written_per_day"]) / (df["likes_given_per_day"] + df["stories_viewed_per_day"] + 1)
df["taux_clic_pub"]      = df["ads_clicked_per_day"] / (df["ads_viewed_per_day"] + 1)
df["ratio_abonnes"]      = df["followers_count"] / (df["following_count"] + 1)
df["sessions_par_heure"] = df["sessions_per_day"] / (df["daily_active_minutes_instagram"] / 60)

RATIOS = ["part_reels", "part_messages", "part_explore", "ratio_createur",
          "taux_clic_pub", "ratio_abonnes", "sessions_par_heure"]
print(len(RATIOS), "ratios créés")

7 ratios créés


In [9]:
filtre3 = pd.DataFrame({
    "moyenne":          df[RATIOS].mean().round(3),
    "coef_variation":   (df[RATIOS].std() / df[RATIOS].mean()).round(2),
    "corr_temps_passe": df[RATIOS].corrwith(df["daily_active_minutes_instagram"]).round(2),
    "corr_stress":      df[RATIOS].corrwith(df["perceived_stress_score"]).round(2),
    "corr_bonheur":     df[RATIOS].corrwith(df["self_reported_happiness"]).round(2),
})
filtre3

,moyenne,coef_variation,corr_temps_passe,corr_stress,corr_bonheur
part_reels,0.255,0.17,0.02,0.01,-0.01
part_messages,0.150,0.23,-0.05,-0.04,0.03
part_explore,0.169,0.25,-0.00,-0.00,0.00
ratio_createur,0.154,0.22,0.36,0.29,-0.14
taux_clic_pub,0.234,0.62,0.06,0.05,-0.02
ratio_abonnes,0.901,0.57,0.01,0.01,-0.01
sessions_par_heure,3.794,0.64,-0.35,-0.28,0.19


In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

for cible in ["perceived_stress_score", "self_reported_happiness"]:
    y = df[cible]
    r2_temps   = cross_val_score(LinearRegression(), df[["daily_active_minutes_instagram"]], y, cv=5, scoring="r2").mean()
    r2_complet = cross_val_score(LinearRegression(), df[["daily_active_minutes_instagram"] + RATIOS], y, cv=5, scoring="r2").mean()
    print(f"{cible} : temps seul = {r2_temps:.3f} | temps + ratios = {r2_complet:.3f}")

perceived_stress_score : temps seul = 0.696 | temps + ratios = 0.696
self_reported_happiness : temps seul = 0.140 | temps + ratios = 0.143


**Filtre 3 – La pertinence.** Nous avons créé 7 ratios qui décrivent la façon d'utiliser Instagram plutôt que la quantité d'usage.

- **Ils apportent une information nouvelle sur l'usage.** Cinq d'entre eux (part des Reels, des messages, d'Explorer, taux de clic publicitaire, ratio abonnés/abonnements) ont une corrélation inférieure à 0,06 avec le temps passé, et un coefficient de variation supérieur à 0,15 : les utilisateurs se différencient réellement sur ces critères.
- **Ils n'apportent rien sur le bien-être.** Le temps passé seul explique 69,6 % des différences de stress déclaré (R², validation croisée en 5 plis). L'ajout des 7 ratios ne change pas ce score (69,6 %). Pour le bonheur, le gain est négligeable (de 14,0 % à 14,3 %).

**Conclusion :** chez les volontaires du panel, le bien-être déclaré est lié à la quantité d'usage, pas à la façon d'utiliser l'application. Les ratios restent utiles pour décrire des profils d'usage destinés aux équipes produit. Il s'agit d'une association statistique, et non d'une relation de cause à effet.

In [11]:
EXCLUES_RGPD = {
    "santé (article 9)": ["perceived_stress_score", "self_reported_happiness", "sleep_hours_per_night",
                          "body_mass_index", "blood_pressure_systolic", "blood_pressure_diastolic",
                          "smoking", "alcohol_frequency", "diet_quality",
                          "exercise_hours_per_week", "daily_steps_count"],
    "vie privée et situation sociale": ["income_level", "employment_status", "education_level",
                                        "relationship_status", "has_children", "weekly_work_hours",
                                        "hobbies_count", "social_events_per_month", "books_read_per_year",
                                        "volunteer_hours_per_month", "travel_frequency_per_year"],
    "profil sensible (mineurs, discrimination, localisation)": ["age", "gender", "country", "urban_rural"],
}

for famille, colonnes in EXCLUES_RGPD.items():
    print(f"{famille} : {len(colonnes)} colonnes exclues")

total = sum(len(c) for c in EXCLUES_RGPD.values())
print(f"\nTotal exclu pour raison RGPD : {total} colonnes sur {df.shape[1]}")

santé (article 9) : 11 colonnes exclues
vie privée et situation sociale : 11 colonnes exclues
profil sensible (mineurs, discrimination, localisation) : 4 colonnes exclues

Total exclu pour raison RGPD : 26 colonnes sur 65


In [12]:
STYLE = ["part_reels", "part_messages", "part_explore", "taux_clic_pub", "ratio_abonnes"]

df[STYLE].describe().round(3)

,part_reels,part_messages,part_explore,taux_clic_pub,ratio_abonnes
count,200000.000,200000.000,200000.000,200000.000,200000.000
mean,0.255,0.150,0.169,0.234,0.901
std,0.044,0.034,0.043,0.146,0.518
min,0.125,0.059,0.059,0.000,0.389
25%,0.220,0.122,0.135,0.143,0.519
50%,0.255,0.150,0.168,0.216,0.717
75%,0.287,0.175,0.202,0.303,1.122
max,0.397,0.273,0.333,7.000,12.597


In [14]:
from sklearn.preprocessing import StandardScaler

borne_basse = df[STYLE].quantile(0.01)
borne_haute = df[STYLE].quantile(0.99)

nb_plafonnes = ((df[STYLE] < borne_basse) | (df[STYLE] > borne_haute)).sum()
print("Valeurs plafonnées par colonne :")
print(nb_plafonnes)

style_plafonne = df[STYLE].clip(lower=borne_basse, upper=borne_haute, axis=1)
X = StandardScaler().fit_transform(style_plafonne)

pd.DataFrame(X, columns=STYLE).describe().round(2)

Valeurs plafonnées par colonne :
part_reels       2623
part_messages    3996
part_explore     3995
taux_clic_pub    1948
ratio_abonnes    4000
dtype: int64


,part_reels,part_messages,part_explore,taux_clic_pub,ratio_abonnes
count,200000.00,200000.00,200000.00,200000.00,200000.00
mean,0.00,-0.00,-0.00,-0.00,0.00
std,1.00,1.00,1.00,1.00,1.00
min,-2.01,-1.90,-1.89,-1.74,-1.00
25%,-0.79,-0.81,-0.80,-0.67,-0.76
50%,0.02,0.02,-0.03,-0.12,-0.36
75%,0.74,0.76,0.78,0.54,0.45
max,2.11,2.13,2.06,3.27,3.03


**Préparation des données pour K-means.** K-means mesure des distances entre utilisateurs. Deux corrections sont donc nécessaires :
- **Plafonnement des valeurs extrêmes.** Certaines valeurs étaient aberrantes : 628 utilisateurs cliquent sur plus de publicités qu'ils n'en voient. Avant correction, le taux de clic montait à 46 écarts-types au-dessus de la moyenne. Nous plafonnons chaque ratio entre son 1er et son 99e centile : entre 1 % et 2 % des valeurs sont concernées selon le ratio, et aucun utilisateur n'est supprimé. Après correction, la valeur maximale tombe à 3,27 écarts-types.
- **Standardisation.** Les ratios n'ont pas la même échelle (la part des Reels va de 0,13 à 0,40, le ratio abonnés de 0,39 à 12,6). Sans correction, les colonnes aux grands chiffres écraseraient les autres. Nous ramenons chaque ratio à une moyenne de 0 et un écart-type de 1.

In [15]:
STYLE    = ["part_reels", "part_messages", "part_explore", "taux_clic_pub", "ratio_abonnes"]
DECRIRE  = ["daily_active_minutes_instagram", "perceived_stress_score",
            "self_reported_happiness", "sleep_hours_per_night"]
VERIFIER = ["age", "gender", "country"]

for nom, liste in [("Construire", STYLE), ("Décrire", DECRIRE), ("Vérifier", VERIFIER)]:
    print(f"{nom} : {len(liste)} colonnes -> {liste}")

Construire : 5 colonnes -> ['part_reels', 'part_messages', 'part_explore', 'taux_clic_pub', 'ratio_abonnes']
Décrire : 4 colonnes -> ['daily_active_minutes_instagram', 'perceived_stress_score', 'self_reported_happiness', 'sleep_hours_per_night']
Vérifier : 3 colonnes -> ['age', 'gender', 'country']


In [16]:
part_mineurs = (df["age"] < 18).mean() * 100
print(f"Part de mineurs dans le fichier : {part_mineurs:.1f} %")

Part de mineurs dans le fichier : 9.5 %


**Rôle des colonnes dans la segmentation.**
- **Construire** : les 5 ratios de style d'usage. Ce sont les seules colonnes utilisées par K-means.
- **Décrire** : le temps passé et les trois indicateurs de bien-être. Ils servent à décrire les familles en moyenne, une fois celles-ci formées.
- **Vérifier** : l'âge, le genre et le pays. Ils ne servent jamais à ranger un utilisateur, mais à contrôler que les familles ne surreprésentent pas les mineurs (9,5 % du fichier) ni ne reproduisent un biais démographique.

In [17]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

echantillon = np.random.RandomState(RANDOM_STATE).choice(len(X), 20000, replace=False)

resultats = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X)
    silhouette = silhouette_score(X[echantillon], km.labels_[echantillon])
    resultats.append({"nb_familles": k, "inertie": round(km.inertia_), "silhouette": round(silhouette, 3)})

choix_k = pd.DataFrame(resultats)
choix_k

,nb_familles,inertie,silhouette
0,2,812186,0.185
1,3,695519,0.178
2,4,597828,0.192
3,5,534367,0.196
4,6,490283,0.178
5,7,464970,0.178
6,8,442775,0.166


In [18]:
choix_k["baisse_inertie"] = choix_k["inertie"].diff()
choix_k

,nb_familles,inertie,silhouette,baisse_inertie
0,2,812186,0.185,NaN
1,3,695519,0.178,-116667.0
2,4,597828,0.192,-97691.0
3,5,534367,0.196,-63461.0
4,6,490283,0.178,-44084.0
5,7,464970,0.178,-25313.0
6,8,442775,0.166,-22195.0


**Choix du nombre de familles.** Nous avons testé K-means de 2 à 8 familles (silhouette calculée sur un échantillon de 20 000 utilisateurs).
- **Inertie** : la baisse est forte jusqu'à 4-5 familles, puis devient faible (−44 084 de 5 à 6, −22 195 de 7 à 8). Le coude se situe autour de 4-5 familles.
- **Silhouette** : elle culmine à 5 familles (0,196), presque à égalité avec 4 familles (0,192), puis redescend.

Les deux solutions se valent statistiquement (écart de 0,004). Nous retenons **4 familles**, plus simples à interpréter et à exploiter par les équipes produit.

**Limite :** une silhouette autour de 0,2 indique des familles qui se chevauchent fortement. Les utilisateurs ne forment pas des groupes nettement séparés, mais glissent progressivement d'un style d'usage à l'autre.

In [20]:
K = 4
modele = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=10).fit(X)
df["famille"] = modele.labels_

(df["famille"].value_counts(normalize=True).sort_index() * 100).round(1)

famille
0    30.5
1    27.2
2    15.1
3    27.3
Name: proportion, dtype: float64

In [21]:
from sklearn.metrics import adjusted_rand_score

for graine in [1, 7, 123]:
    autre = KMeans(n_clusters=K, random_state=graine, n_init=10).fit(X)
    ari = adjusted_rand_score(modele.labels_, autre.labels_)
    print(f"Graine {graine} : ARI = {ari:.3f}")

Graine 1 : ARI = 0.994
Graine 7 : ARI = 0.990
Graine 123 : ARI = 0.976


**Construction et stabilité des familles.** Nous construisons 4 familles avec K-means. La plus petite regroupe 15,1 % des utilisateurs (environ 30 000 personnes) : aucune famille n'isole une poignée d'utilisateurs.

**Stabilité.** Relancer K-means avec trois autres graines aléatoires (1, 7 et 123) donne des familles quasi identiques : ARI de 0,994, 0,990 et 0,976 (1 = familles identiques). Les familles ne dépendent donc pas du hasard du point de départ.

**Interprétation.** Les familles sont stables mais se chevauchent (silhouette de 0,19) : elles décrivent des tendances d'usage solides, avec de nombreux utilisateurs à la frontière entre deux styles.

In [22]:
centres = pd.DataFrame(modele.cluster_centers_, columns=STYLE).round(2)
centres["part_utilisateurs_%"] = (df["famille"].value_counts(normalize=True).sort_index() * 100).round(1)
centres

,part_reels,part_messages,part_explore,taux_clic_pub,ratio_abonnes,part_utilisateurs_%
0,0.88,-0.07,-0.87,-0.02,-0.35,30.5
1,-0.11,-0.87,0.90,0.05,-0.33,27.2
2,0.02,0.02,-0.02,-0.00,1.93,15.1
3,-0.89,0.93,0.08,-0.03,-0.34,27.3


In [23]:
NOMS = {0: "Adeptes des Reels", 1: "Explorateurs", 2: "Comptes très suivis", 3: "Messagers"}
df["nom_famille"] = df["famille"].map(NOMS)
df["nom_famille"].value_counts(normalize=True).round(3) * 100

nom_famille
Adeptes des Reels      30.5
Messagers              27.3
Explorateurs           27.2
Comptes très suivis    15.1
Name: proportion, dtype: float64

In [24]:
description = df.groupby("nom_famille")[DECRIRE].mean().round(2)
description["part_mineurs_%"] = (df["age"].lt(18).groupby(df["nom_famille"]).mean() * 100).round(1)
description["part_femmes_%"] = (df["gender"].eq("Female").groupby(df["nom_famille"]).mean() * 100).round(1)
description

,daily_active_minutes_instagram,perceived_stress_score,self_reported_happiness,sleep_hours_per_night,part_mineurs_%,part_femmes_%
nom_famille,,,,,,
Adeptes des Reels,184.66,19.65,5.55,7.00,9.2,48.4
Comptes très suivis,191.85,20.31,5.46,6.99,9.7,47.6
Explorateurs,198.75,20.87,5.33,7.00,9.7,47.7
Messagers,181.56,19.43,5.60,7.00,9.3,47.9


In [26]:
from sklearn.metrics import r2_score

df["groupe_temps"] = pd.qcut(df["daily_active_minutes_instagram"], 3,
                             labels=["petit", "moyen", "gros"])

y = df["perceived_stress_score"]
r2_familles = r2_score(y, df.groupby("nom_famille")["perceived_stress_score"].transform("mean"))
r2_temps    = r2_score(y, df.groupby("groupe_temps", observed=True)["perceived_stress_score"].transform("mean"))
print(f"Part du stress expliquée par les 4 familles de style : {r2_familles:.1%}")
print(f"Part du stress expliquée par les 3 groupes de temps passé : {r2_temps:.1%}")

pd.crosstab(df["nom_famille"], df["groupe_temps"], normalize="index").round(3) * 100

Part du stress expliquée par les 4 familles de style : 0.2%
Part du stress expliquée par les 3 groupes de temps passé : 63.3%


groupe_temps,petit,moyen,gros
nom_famille,,,
Adeptes des Reels,34.8,32.6,32.6
Comptes très suivis,32.4,33.3,34.3
Explorateurs,29.6,35.4,35.0
Messagers,35.9,32.2,31.9


In [27]:
scaler = StandardScaler().fit(style_plafonne)
print("Même standardisation que X :", np.allclose(scaler.transform(style_plafonne), X))

Même standardisation que X : True


In [28]:
def affecter_famille(utilisateur):
    """Prend les données d'usage brutes d'un utilisateur et renvoie le nom de sa famille."""
    # 1. Calculer ses 5 ratios
    temps_total = (utilisateur["time_on_feed_per_day"] + utilisateur["time_on_explore_per_day"]
                   + utilisateur["time_on_messages_per_day"] + utilisateur["time_on_reels_per_day"])
    ratios = pd.DataFrame([{
        "part_reels":    utilisateur["time_on_reels_per_day"] / temps_total,
        "part_messages": utilisateur["time_on_messages_per_day"] / temps_total,
        "part_explore":  utilisateur["time_on_explore_per_day"] / temps_total,
        "taux_clic_pub": utilisateur["ads_clicked_per_day"] / (utilisateur["ads_viewed_per_day"] + 1),
        "ratio_abonnes": utilisateur["followers_count"] / (utilisateur["following_count"] + 1),
    }])
    # 2. Plafonner avec les limites du fichier d'origine
    ratios = ratios.clip(lower=borne_basse, upper=borne_haute, axis=1)
    # 3. Standardiser avec la règle graduée du fichier d'origine
    ratios_standardises = scaler.transform(ratios)
    # 4. Trouver l'arrêt de bus le plus proche
    numero = modele.predict(ratios_standardises)[0]
    return NOMS[numero]

In [29]:
profil_type = {
    "time_on_feed_per_day": 40, "time_on_explore_per_day": 15,
    "time_on_messages_per_day": 15, "time_on_reels_per_day": 50,
    "ads_viewed_per_day": 20, "ads_clicked_per_day": 3,
    "followers_count": 300, "following_count": 400,
}

profils_fictifs = {
    "Beaucoup de Reels":           profil_type,
    "Beaucoup de messagerie":      {**profil_type, "time_on_reels_per_day": 20, "time_on_messages_per_day": 40},
    "Beaucoup d'abonnés":          {**profil_type, "followers_count": 5000, "time_on_reels_per_day": 30, "time_on_explore_per_day": 20},
    "Beaucoup d'exploration":      {**profil_type, "time_on_reels_per_day": 25, "time_on_explore_per_day": 35},
}

for description, profil in profils_fictifs.items():
    print(f"{description:25} -> {affecter_famille(profil)}")

Beaucoup de Reels         -> Adeptes des Reels
Beaucoup de messagerie    -> Messagers
Beaucoup d'abonnés        -> Comptes très suivis
Beaucoup d'exploration    -> Explorateurs


## Affecter un nouvel utilisateur

La fonction `affecter_famille` range n'importe quel utilisateur d'Instagram dans l'une des 4 familles, à partir de ses seules données d'usage (temps par section, publicités vues et cliquées, abonnés et abonnements). Elle n'utilise aucune donnée de santé, ni l'âge, ni le genre : elle s'applique donc à tous les utilisateurs, et pas seulement aux volontaires du panel.

Le nouvel utilisateur suit exactement le même chemin que les 200 000 utilisateurs du fichier : calcul des 5 ratios, plafonnement avec les limites du fichier, standardisation avec la moyenne et l'écart-type du fichier, puis affectation à la famille dont le centre est le plus proche. La démonstration utilise des profils inventés, afin de n'afficher aucun enregistrement réel.